This notebook tries to memorize a trajectory by learning a policy to predict the next point given the previous 64 points.

It doesn't really work.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#@markdown ### **Imports**
# diffusion policy import
from typing import Tuple, Sequence, Dict, Union, Optional
import numpy as np
import math
import torch
import torch.nn as nn
import collections
import zarr
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_scheduler
from tqdm.auto import tqdm

# Painting imports
import cv2
from style.diffusion_policy_gml.dataset import PushTStateDataset
from style.diffusion_policy_gml.network import MemorizationModel, MemorizationModel2, ConditionalUnet1D
from style.diffusion_policy_gml.env import PaintingEnv
import style.diffusion_policy_gml.network as network
import matplotlib.pyplot as plt

In [ ]:
#@markdown ### **Dataset Demo**

# use cached dataset
dataset_path = "data/gml_000000.zarr"
# dataset_path = "data/gml_003000.zarr"

# parameters
pred_horizon = 64
obs_horizon = 64
action_horizon = 1
#|o|o|                             observations: 2
#| |a|a|a|a|a|a|a|a|               actions executed: 8
#|p|p|p|p|p|p|p|p|p|p|p|p|p|p|p|p| actions predicted: 16

# create dataset from file
dataset = PushTStateDataset(
    dataset_path=dataset_path,
    pred_horizon=pred_horizon,
    obs_horizon=obs_horizon,
    action_horizon=action_horizon,
    action_delta=True
)

print(np.argwhere(dataset.indices[:, 1] - dataset.indices[:, 0] == 132))
print(dataset.indices[256])
print(dataset.indices.shape)
# dataset.indices = dataset.indices[[256]]
# Only grab the drawings (episodes) starting at t=301 (the next episode starts at 433)
dataset.indices = dataset.indices[dataset.indices[:, 0] >= 301]
dataset.indices = dataset.indices[dataset.indices[:, 0] < 433]
print(dataset.indices.shape)

# create dataloader
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=256,
    num_workers=1,
    shuffle=True,
    # accelerate cpu-gpu transfer
    pin_memory=True,
    # don't kill worker process afte each epoch
    persistent_workers=True
)

# visualize data in batch
print(len(list(iter(dataloader))))
batch = next(iter(dataloader))
print("batch['obs'].shape:", batch['obs'].shape)
print("batch['action'].shape", batch['action'].shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for b in range(0, batch['obs'].shape[0], 10):
    axes[0].plot(batch['obs'][b, :, 0] + b / 250, batch['obs'][b, :, 1], 'o-')
for b in range(0, batch['obs'].shape[0], 10):
    axes[0].plot(batch['action'][b, :, 0] + b / 250, batch['action'][b, :, 1], '.-')
axes[0].axis('equal')
# axes[1].axis('equal')
# plt.plot(dataset.normalized_train_data['obs'][:, 0], dataset.normalized_train_data['obs'][:, 1], '.-')
# fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
# for i in range(3):
#     axes[i].plot(dataset[i + 13]['obs'][:, 0], dataset[i + 13]['obs'][:, 1], '.-')

# Plot the segments
xy = dataset.normalized_train_data['obs']
starts = []
ends = []
for b in range(0, batch['obs'].shape[0], 10):
    # Find where batch['obs'][b, 0, :] is in xy
    starts.append(np.argwhere((xy == batch['obs'][b, 0].numpy()).all(axis=1))[0, 0])
    ends.append(np.argwhere((xy == batch['obs'][b, -1].numpy()).all(axis=1))[0, 0])
starts, ends = np.array(starts), np.array(ends)
# Create a vertical bar chart going from buffer_start_idx to buffer_end_idx
xs = list(range(0, batch['obs'].shape[0], 10))
# axes[1].bar(xs, dataset.indices[xs, 1] - dataset.indices[xs, 0], width=1, bottom=dataset.indices[xs, 0], color='r', alpha=0.5)
axes[1].bar(xs, ends - starts, width=1, bottom=starts, color='r', alpha=0.5)

In [ ]:
# for this demo, we use DDPMScheduler with 100 diffusion iterations
num_diffusion_iters = 100
noise_scheduler = DDPMScheduler(
    num_train_timesteps=num_diffusion_iters,
    # the choise of beta schedule has big impact on performance
    # we found squared cosine works the best
    beta_schedule='squaredcos_cap_v2',
    # clip output to [-1,1] to improve stability
    clip_sample=False,
    clip_sample_range=5,
    # our network predicts noise (instead of denoised action)
    prediction_type='epsilon'
)

In [ ]:
#@markdown ### **Network Demo**

# observation and action dimensions corrsponding to
# the output of PushTEnv
obs_dim = 2
action_dim = 2

# create network object
noise_pred_net = MemorizationModel2(
    input_dim=action_dim,
    global_cond_dim=obs_dim*obs_horizon,
    noise_scheduler=noise_scheduler,
    T=64
)

In [ ]:
# Check the network doesn't throw syntax errors
# example inputs
noised_action = torch.randn((1, pred_horizon, action_dim))
print('Diffusion Action Shape:', noised_action.shape)
obs = torch.zeros((1, obs_horizon, obs_dim))
print('Conditioning Shape:', obs.shape)
diffusion_iter = torch.zeros((1,), dtype=torch.long)

# the noise prediction network
# takes noisy action, diffusion iteration and observation as input
# predicts the noise added to action
noise = noise_pred_net(
    sample=noised_action,
    timestep=diffusion_iter,
    global_cond=obs.flatten(start_dim=1))

# illustration of removing noise
# the actual noise removal is performed by NoiseScheduler
# and is dependent on the diffusion noise schedule
denoised_action = noised_action - noise
print('Denoised Action Shape:', denoised_action.shape)

# device transfer
device = torch.device('cuda')
_ = noise_pred_net.to(device)

In [ ]:
# Initialize `noise_pred_net` with random weights
def init_weights(m):
    if type(m) == nn.Linear:
        torch.nn.init.xavier_uniform_(m.weight)
        m.bias.data.fill_(0.01)
noise_pred_net.apply(init_weights)

In [ ]:
#@markdown ### **Training**

num_epochs = 2500

# Exponential Moving Average
# accelerates training and improves stability
# holds a copy of the model weights
ema = EMAModel(
    parameters=noise_pred_net.parameters(),
    model=noise_pred_net,
    power=0.75)

# Standard ADAM optimizer
# Note that EMA parametesr are not optimized
optimizer = torch.optim.AdamW(
    params=noise_pred_net.parameters(),
    lr=1e-4, weight_decay=1e-6)

# Cosine LR schedule with linear warmup
lr_scheduler = get_scheduler(
    name='cosine',
    optimizer=optimizer,
    num_warmup_steps=500,
    num_training_steps=len(dataloader) * num_epochs
)

with tqdm(range(num_epochs), desc='Epoch') as tglobal:
    # epoch loop
    loss_history = list()
    for epoch_idx in tglobal:
        epoch_loss = list()
        # batch loop
        # with tqdm(dataloader, desc='Batch', leave=False) as tepoch:
        tepoch = dataloader
        if True:
            for nbatch in tepoch:
                # data normalized in dataset
                # device transfer
                nobs = nbatch['obs'].to(device)
                naction = nbatch['action'].to(device)
                B = nobs.shape[0]

                # observation as FiLM conditioning
                # (B, obs_horizon, obs_dim)
                obs_cond = nobs[:,:obs_horizon,:]
                # (B, obs_horizon * obs_dim)
                obs_cond = obs_cond.flatten(start_dim=1)

                # sample noise to add to actions
                noise = torch.randn(naction.shape, device=device)

                # sample a diffusion iteration for each data point
                timesteps = torch.randint(
                    0, noise_scheduler.config.num_train_timesteps,
                    (B,), device=device
                ).long()

                # add noise to the clean images according to the noise magnitude at each diffusion iteration
                # (this is the forward diffusion process)
                noisy_actions = noise_scheduler.add_noise(
                    naction, noise, timesteps)

                # predict the noise residual
                noise_pred = noise_pred_net(
                    noisy_actions, timesteps, global_cond=obs_cond)

                # L2 loss
                loss = nn.functional.mse_loss(noise_pred, noise)

                # optimize
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                # step lr scheduler every batch
                # this is different from standard pytorch behavior
                lr_scheduler.step()

                # update Exponential Moving Average of the model weights
                ema.step(noise_pred_net)

                # logging
                loss_cpu = loss.item()
                epoch_loss.append(loss_cpu)
                # tepoch.set_postfix(loss=loss_cpu)
        tglobal.set_postfix(loss=np.mean(epoch_loss))
        loss_history.append(np.mean(epoch_loss))

# Weights of the EMA model
# is used for inference
# ema_noise_pred_net = ema.averaged_model
plt.semilogy(loss_history)

In [ ]:
# Print random training samples with increasing levels of noise
a = np.zeros((10, *noisy_actions.shape[1:]))
b = np.zeros((10, *naction.shape[1:]))
c = np.zeros((10, *noise_pred.shape[1:]))
timesteps_to_plot = []
for i in range(10):
    ind = (timesteps - num_diffusion_iters * i / 9).abs().argmin().item()
    print(timesteps[ind].item(), end=', ')
    a[i] = noisy_actions[ind].cpu().numpy()
    b[i] = naction[ind].cpu().numpy()
    with torch.no_grad():
        predicted_noise = noise_pred[ind][None, ...]
        c[i] = network.compute_orig(noise_scheduler, timesteps[ind][None, ...], noisy_actions[ind][None, ...], predicted_noise).cpu().numpy()
    timesteps_to_plot.append(timesteps[ind].item())
print()
fig, axes = plt.subplots(2, 5, figsize=(20, 5), sharex=True, sharey=True)
for i, ax in enumerate(axes.flatten()):
    ax.plot(b[i, :, 0], b[i, :, 1], 'k.-')
    ax.plot(a[i, :, 0], a[i, :, 1], 'r.-', markersize=2, linewidth=0.5)
    ax.plot(c[i, :, 0], c[i, :, 1], 'b.-', markersize=2, linewidth=0.5)
    ax.set_title(f'Timestep: {timesteps_to_plot[i]}')
fig.suptitle('Training Examples: Noisy Actions');

In [ ]:
ema_noise_pred_net = MemorizationModel2(
    input_dim=action_dim,
    global_cond_dim=obs_dim*obs_horizon,
    noise_scheduler=noise_scheduler,
    T=64
)
ema_noise_pred_net.to(device)
ema.copy_to(ema_noise_pred_net.parameters())

In [ ]:
# Initialize the environment
starting_point = dataset[10]['obs'][0]
print(starting_point)
obs, info = env.reset(x0=dataset.unnormalize_obs(starting_point))
# print(action)
# obs, *_ = env.step(dataset.unnormalize_action(np.array([0, 0])))
for action in dataset[10]['action'][:-1]:
    obs, *_ = env.step(dataset.unnormalize_action(action))
# plt.plot(*dataset.normalize_obs(np.array(env.traj)).T, 'r.-')
# plt.plot(*dataset[10]['obs'].T, 'k.-');
plt.plot(dataset.normalize_obs(np.array(env.traj)), 'r.-')
plt.plot(dataset[10]['obs'], 'k.-');

In [ ]:
#@markdown ### **Inference**

# limit enviornment interaction to 200 steps before termination
max_steps = 200
env = PaintingEnv(resolution=(512, 512))
# use a seed >200 to avoid initial states seen in the training dataset
# env.seed(100000)

# get first observation
obs, info = env.reset(x0=dataset.unnormalize_obs(starting_point))
print(obs)

# keep a queue of last `obs_horizon` steps of observations
obs_deque = collections.deque([obs] * obs_horizon, maxlen=obs_horizon)
# save visualization and rewards
imgs = [env.render(mode='rgb_array')]
rewards = list()
done = False
step_idx = 0

all_actions = []
all_obs = []

with tqdm(total=max_steps, desc="Eval PushTStateEnv") as pbar:
    while not done:
        B = 1
        # stack the last obs_horizon (2) number of observations
        obs_seq = np.stack(obs_deque)
        # normalize observation
        nobs = dataset.normalize_obs(obs_seq)
        all_obs.append(nobs)
        # device transfer
        nobs = torch.from_numpy(nobs).to(device, dtype=torch.float32)

        # infer action
        with torch.no_grad():
            # reshape observation to (B,obs_horizon*obs_dim)
            obs_cond = nobs.unsqueeze(0).flatten(start_dim=1)

            # initialize action from Guassian noise
            noisy_action = torch.randn(
                (B, pred_horizon, action_dim), device=device)
            # Set the first portion of the action to the observation
            noisy_action[0, :obs_horizon] = nobs
            naction = noisy_action

            # init scheduler
            noise_scheduler.set_timesteps(num_diffusion_iters)

            tmp = [naction.detach().to('cpu').numpy()]
            for k in noise_scheduler.timesteps:
                # predict noise
                noise_pred = ema_noise_pred_net(
                    sample=naction,
                    timestep=k,
                    global_cond=obs_cond
                )

                # inverse diffusion step (remove noise)
                naction = noise_scheduler.step(
                    model_output=noise_pred,
                    timestep=k,
                    sample=naction
                ).prev_sample

                tmp.append(naction.detach().to('cpu').numpy())
            all_actions.append(tmp)
            # raise Exception

        # unnormalize action
        naction = naction.detach().to('cpu').numpy()
        # (B, pred_horizon, action_dim)
        naction = naction[0] # batch is only 1 during inference, so squeeze
        action_pred = dataset.unnormalize_action(naction)

        # only take action_horizon number of actions
        start = obs_horizon - 1
        # end = start + action_horizon
        end = start + action_horizon
        action = action_pred[start:end,:]
        assert len(action) == action_horizon
        # (action_horizon, action_dim)

        # execute action_horizon number of steps
        # without replanning
        for i in range(len(action)):
            # stepping env
            obs, reward, done, _, info = env.step(action[i], delta=True)
            # save observations
            obs_deque.append(obs)
            # and reward/vis
            rewards.append(reward)
            # imgs.append(env.render(mode='rgb_array'))

            # update progress bar
            step_idx += 1
            pbar.update(1)
            pbar.set_postfix(reward=reward)
            if step_idx > max_steps:
                done = True
            if done:
                break

# print out the maximum target coverage
print('Score: ', max(rewards))

# # visualize
# from IPython.display import Video
# from skvideo.io import vwrite
# vwrite('vis.mp4', imgs, outputdict={'-pix_fmt': 'yuv420p'})
# Video('vis.mp4', embed=True, width=256, height=256)
all_actions = np.array(all_actions)
all_obs = np.array(all_obs)

In [ ]:
tr = np.array(env.traj)
gt = dataset.unnormalize_obs(dataset[-1]['obs'])
gt = dataset.unnormalize_obs(dataset.normalized_train_data['obs'][301:433])
print(gt.shape)
plt.plot(gt[:, 0], gt[:, 1], 'k.-', label='Ground Truth')
plt.plot(tr[:, 0], tr[:, 1], 'r.-', label='Diffusion Policy')
plt.title('Final Trajectory')
plt.legend();

In [ ]:
print(all_obs.shape) # (policy timestep, T, xy)
print(all_actions.shape) # (policy timestep, diffusion timestep, batch, T, xy)

In [ ]:
# First plot the very first action diffusion process
obs0 = all_obs[0]
tmp0 = all_actions[0][:, 0, :, :]
fig, axes = plt.subplots(2, 5, figsize=(15, 5), sharex=True, sharey=True)
axes = axes.flatten()
for i in range(10):
    axes[i].plot(obs0[:, 0], obs0[:, 1], 'r*-', markersize=10)
    axes[i].plot(tmp0[i*11, :, 0], tmp0[i*11, :, 1], '.-', alpha=0.3)
axes[-1].plot(tmp0[-1, -1, 0], tmp0[-1, -1, 1], 'k*')

In [ ]:
# Now plot the trajectory as it's evolving
plt.plot(all_obs[:, -1, 0], all_obs[:, -1, 1], 'r.-')